# Ollama에서 Hugging Face GGUF 모델 사용하기

Hugging Face에 공개된 GGUF 모델은 파일을 별도 폴더에 내려받고 `Modelfile`로 등록하지 않아도 `hf.co/{사용자}/{저장소}:{양자화}` 형식으로 Ollama에 가져올 수 있다. 이 노트북에서는 모델을 Ollama cache에 한 번 준비한 뒤 Python API와 LangChain에서 같은 model ID를 재사용한다.


## GGUF 포맷

[GGUF](https://huggingface.co/docs/hub/en/gguf)는 모델 가중치뿐 아니라 tokenizer와 실행에 필요한 metadata를 하나의 파일에 담는 포맷이다. llama.cpp와 Ollama 같은 경량 추론 엔진이 빠르게 읽을 수 있으며, 여러 양자화 방식을 지원한다.

- **단일 파일**: 가중치와 metadata를 함께 보관해 배포하기 쉽다.
- **양자화 지원**: 4비트·5비트처럼 낮은 정밀도로 저장해 파일 크기와 추론 메모리를 줄일 수 있다.
- **실행 엔진 호환**: llama.cpp 계열 도구와 Ollama에서 사용할 수 있다.

`Q5_K_M`은 5비트 계열 양자화 방식이다. 일반적으로 더 낮은 bit 수는 메모리를 줄이지만 원본 가중치와의 차이가 커질 수 있다.

## 선수 조건과 패키지 준비

RunPod에서 `01_ollama.ipynb`를 먼저 완료해 Ollama server가 실행 중이어야 한다. Python의 `ollama` package는 새 server를 만드는 도구가 아니라 같은 Pod의 `http://localhost:11434` server에 요청하는 client이다.

In [1]:
%pip install -U ollama langchain-ollama

import ollama
from langchain_ollama import ChatOllama


  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 2.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 5.8 MB/s  0:00:00
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 33.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [lang

## Hugging Face GGUF 모델 바로 사용하기

`heegyu/EEVE-Korean-Instruct-10.8B-v1.0-GGUF:latest` 저장소는 현재 Ollama의 Hugging Face manifest 처리에서 호환 오류가 발생한다. 따라서 같은 `yanolja/EEVE-Korean-Instruct-10.8B-v1.0` 기반의 `Q5_K_M` GGUF인 [SourPineapple 저장소](https://huggingface.co/SourPineapple/EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF)를 사용한다.

`ollama pull`은 GGUF를 임의의 `/workspace` 폴더에 저장하는 명령이 아니다. Ollama가 관리하는 model cache에 최초 한 번 내려받아 이후 `generate()`와 `chat()`이 같은 model ID를 사용할 수 있게 한다. [Hugging Face의 Ollama 가이드](https://huggingface.co/docs/hub/en/ollama)는 저장소 뒤에 `:Q5_K_M`처럼 tag를 붙여 원하는 양자화를 선택하는 형식을 제공한다.

In [2]:
MODEL_ID = (
    'hf.co/SourPineapple/'
    'EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF:Q5_K_M'
)


### Ollama cache에 모델 준비하기

Python client의 `pull()`에 model ID를 전달하면 Ollama server가 Hugging Face에서 GGUF를 내려받아 자체 cache로 관리한다. 최초 실행은 파일 크기만큼 시간이 필요하지만 이후 실행에서는 저장된 layer를 재사용한다.

In [3]:
import ollama

pull_result = ollama.pull(MODEL_ID)

print(pull_result)

status='success' completed=None total=None digest=None


## `ollama.generate()`로 한 번 생성하기

`generate()`는 하나의 prompt를 전달해 assistant 역할 구분 없이 텍스트를 생성한다. `model`과 `prompt`를 전달하며, 반환값의 `response`에 생성된 본문이 들어 있다.

In [5]:
response = ollama.generate(
    model = MODEL_ID,
    prompt = "Meta 회사의 Ollama가 뭐야?"
)

print(response["response"])

Meta의 올라마는 머신 러닝을 활용하여 사람들의 소셜 미디어 활동을 분석하고 개인 맞춤형 콘텐츠, 제품, 서비스를 추천하는 AI 기반 시스템입니다. 이는 Meta의 플랫폼에서 참여도를 높이고 사용자 경험을 향상시키기 위해 설계되었습니다.

올라마는 사람들의 소셜 미디어 활동, 좋아요, 댓글, 공유된 콘텐츠, 그리고 상호작용한 친구들을 분석하여 그들의 취향과 관심사를 이해합니다. 이러한 데이터를 바탕으로 올라마는 사용자가 좋아할 만한 관련 콘텐츠, 제품 또는 서비스를 개인 맞춤형으로 제안하여 Meta 플랫폼에서의 사용자 경험을 향상시킬 수 있습니다. 올라마는 또한 사용자의 선호에 따라 광고 타겟팅 및 맞춤화에도 사용될 수 있어 광고의 관련성을 높이고 사용자 참여를 높이는 데 도움이 됩니다.

요약하자면, Meta의 올라마는 머신 러닝을 활용하여 사람들의 소셜 미디어 활동을 분석하여 개인 맞춤형 콘텐츠, 제품, 서비스를 제공하고 참여도를 증가시키며 사용자 경험을 향상시키는 AI 기반 시스템입니다.


## `ollama.chat()`으로 역할이 있는 대화하기

`chat()`은 `system`, `user`, `assistant` 역할이 있는 message 목록을 전달한다. 반환값의 `message.content`에서 assistant 답변을 꺼낸다.

In [11]:
chat_response = ollama.chat(
    model = MODEL_ID,
    messages=[
        {"role" : "system", "content" : "너는 현업 AI 엔지니어로 일하고 있고, 조언을 해주는 역할이야. 관련 질문에 대해서 간략하게 앙큼한 깜찍한 2문단 이내로 대답해."},
        {"role" : "user", "content" : "신입 AI 엔지니어가 되려면 무엇을 준비해야해?"}
    ]
)

print(chat_response["message"]["content"])

신입 AI 엔지니어가 되려면 다음 사항을 준비하면 됩니다:

1. **컴퓨터 과학 기초 마스터하기**: 인공지능(AI)은 주로 데이터 과학, 컴퓨터 비전, 자연어 처리와 같은 컴퓨터 과학 분야에서 파생됩니다. 프로그래밍 언어(파이썬, 자바, C++), 알고리즘, 데이터 구조에 대한 탄탄한 이해를 개발하세요.

2. **AI 관련 기술 배우기**: 머신 러닝, 딥 러닝, 신경망, 자연어 처리, 컴퓨터 비전과 같은 AI 분야에 몰두하세요. 이를 위해 온라인 강좌, MOOC(대규모 개방형 온라인 코스), 책, AI 전문 온라인 커뮤니티에서 제공하는 자료를 활용하세요.

3. **프로젝트 참여하기**: AI에 대한 이해를 깊게 하고 기술을 연마하기 위해 프로젝트를 진행하세요. GitHub와 같은 플랫폼에서 프로젝트를 공유하여 포트폴리오를 구축하고 관련 커뮤니티의 주목을 받으세요.

4. **관련 경력 쌓기**: AI와 관련된 인턴십, 연구 보조원직 또는 자원봉사 활동을 찾아보세요. 경험을 쌓으면서 해당 분야의 네트워크를 확장하고 잠재적 고용주를 만날 수 있습니다.

5. **자격증 취득하기**: 컴퓨터 과학이나 관련 분야에서 학사 학위나 석사 학위를 취득하세요. AI에 특화되어 있다면 더 경쟁력 있는 후보자가 될 수 있습니다.

6. **소프트웨어 도구 숙련**: AI 프로젝트 개발에 사용되는 인기 소프트웨어 도구와 플랫폼, 예를 들어 TensorFlow, PyTorch, Keras, OpenCV, NLTK 등에 능숙해지세요.

7. **업계 동향에 민감하게 대응하기**: AI 분야는 빠르게 발전하고 있으므로 최신 트렌드, 혁신, 연구 결과에 대해 정보를 얻기 위해 업계 뉴스를 따라가세요.

8. **커뮤니케이션 기술 개발하기**: AI 엔지니어는 복잡한 개념을 비기술적 이해 관계자에게 전달할 수 있어야 합니다. 글쓰기, 구두 발표, 프레젠테이션 기술에 집중하세요.

9. **팀워크 능력 향상하기**: AI 프로젝트는 종종 다양한 배경을 가진 사람들로 구성된 다학제 

## LangChain `ChatOllama`로 같은 모델 호출하기

`ChatOllama`는 같은 Ollama server와 model ID를 LangChain의 Chat Model 인터페이스로 감싼다. GGUF를 다시 내려받거나 다른 모델로 변환하는 과정과 관계가 없다.

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = MODEL_ID,
    temperature = 0.2
)

langchain_response = llm.invoke("아프로 켄이 뭐야?")

print(langchain_response.content)

아프로 켄은 아프리카의 다양한 문화와 역사를 탐구하는 교육적이고 재미있는 애니메이션 시리즈입니다. 이 시리즈는 아프리카의 풍부한 유산과 다양한 문화, 전통, 신념을 젊은 시청자들에게 소개하는 것을 목표로 합니다.

아프로 켄은 아프리카 대륙을 배경으로 하며, 다양한 동물 캐릭터들이 등장하여 자신들의 이야기와 지식을 공유합니다. 이 캐릭터들은 아프리카의 다양한 동물들을 대표하며, 각 에피소드마다 새로운 동물이 등장하여 자신들의 문화, 전통, 신념을 소개합니다.

시리즈의 주요 캐릭터로는 아프리카 코끼리인 아프로, 아프리카 사자인 켄, 그리고 다양한 동물 캐릭터들이 있습니다. 이 캐릭터들은 자신들의 이야기와 지식을 공유하며, 아프리카의 역사, 문화, 전통에 대해 젊은 시청자들에게 교육합니다.

아프로 켄은 아프리카의 다양한 문화와 역사를 젊은 시청자들에게 소개하는 것뿐만 아니라, 환경 보호, 지속 가능성, 문화 다양성에 대한 중요성도 강조합니다. 시리즈는 아프리카의 자연 아름다움과 생물 다양성을 보여주며, 젊은 시청자들이 환경에 대한 책임감을 가지도록 영감을 줍니다.

아프로 켄은 젊은 시청자들에게 아프리카의 풍부한 유산과 다양한 문화, 전통, 신념을 소개하는 재미있고 교육적인 시리즈입니다. 시리즈는 아프리카의 역사, 문화, 전통에 대해 젊은 시청자들에게 교육하는 것뿐만 아니라, 환경 보호, 지속 가능성, 문화 다양성에 대한 중요성도 강조합니다.


## 선택 확장: 직접 `Modelfile`을 만드는 경우

두 번째 경로인 `GGUF 다운로드 → Modelfile 작성 → ollama create`는 model의 prompt template와 생성 parameter를 직접 바꿀 때만 사용한다. 기본 실습처럼 공개 GGUF를 그대로 실행할 때는 필요하지 않으므로 이 노트북의 필수 실행 단계에서 제외한다. 변환과 수동 등록 과정은 `hf_to_gguf.ipynb`에서 별도로 다룬다.

## 정리

- 기본 실습은 Hugging Face model ID를 Ollama cache에 준비하고 `ollama.generate()`로 호출한다.
- `/workspace`에 GGUF를 따로 내려받는 과정은 기본 실습에서 제외한다.
- `generate()`, `chat()`, `ChatOllama`는 같은 Ollama server와 같은 model을 서로 다른 입력 형식으로 호출한다.
- `Modelfile`은 공개 GGUF의 template나 parameter를 직접 바꿔야 할 때만 선택한다.